# 06 (Kaggle) — Ask the model a question, get Verilog back

A live web page: type a spec ("4-bit synchronous up counter with
active-low reset"), get a generated Verilog module back. Backed by
`scripts/ask.py`'s logic and the same (now-fixed) quantized model loading
`generate.py` uses -- the model is loaded **once** in this notebook and
reused for every question, rather than reloading per question the way the
plain CLI script does.

**Before running:** attach the Kaggle Dataset for whichever trained
adapter you want to serve (`verilog-slm-m1-final` for M1, or a fresh
`verilog-slm-m0-final` once M0 is retrained). Accelerator = GPU,
Internet = On.

**About the link this produces:** Gradio's `share=True` gives a public
URL (via Gradio's own tunnel, not Kaggle) that works in any browser for
about 72 hours -- anyone with the link can use it, there's no login by
default. It also stops working the moment this Kaggle session ends,
same as everything else in this project that isn't explicitly saved.
If you want a password on it, see the commented `auth=` line in the
launch cell below.

In [ ]:
# --- Bootstrap: repo + the adapter to serve ---
import os, shutil, glob

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/verilog-slm'):
    os.system(f'git clone {REPO} /kaggle/working/verilog-slm')
os.chdir('/kaggle/working/verilog-slm')
os.system('git pull')
os.makedirs('artifacts', exist_ok=True)

# ADAPTER_KEYWORD selects which trained model to serve -- match it to
# whichever dataset you attached (e.g. 'm1-final' or 'm0-final').
ADAPTER_KEYWORD = "m1-final"
ADAPTER_DIR = "artifacts/serve_adapter"

cfgs = [p for p in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True) if ADAPTER_KEYWORD in p]
if not cfgs:
    raise SystemExit(f'No adapter found with "{ADAPTER_KEYWORD}" in its path -- attach that dataset first.')
src = os.path.dirname(sorted(cfgs, key=len)[0])
shutil.rmtree(ADAPTER_DIR, ignore_errors=True)
shutil.copytree(src, ADAPTER_DIR)
print(f'serving adapter: {ADAPTER_DIR} <- {src}')
print('files:', sorted(os.listdir(ADAPTER_DIR)))

In [ ]:
!pip install -q -r requirements.txt -r requirements-train.txt
!pip install -q -U "torchao>=0.16.0" gradio
print('installs done')

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
!nvidia-smi -L

## Load the model once

This is the slow step (quantized load, a minute or two) -- it only runs
once per session, not per question.

In [ ]:
import sys
sys.path.insert(0, '.')
from src.infer.generate import load_model_for_inference, generate_batch
from src.infer.postprocess import finalize
from src.utils.prompts import build_prompt
from src.utils.seeding import set_seed

print('loading model...')
model, tokenizer = load_model_for_inference(ADAPTER_DIR)
print('model loaded and ready')

## The Q&A page

In [ ]:
import gradio as gr

def answer(question: str, temperature: float, seed: int) -> str:
    if not question or not question.strip():
        return "Enter a specification above, e.g. \"4-bit synchronous up counter with active-low reset\"."
    set_seed(int(seed))
    prompt = build_prompt(question.strip())
    [[completion]] = generate_batch(model, tokenizer, [prompt], n=1, temperature=temperature, top_p=0.95)
    return finalize(completion)

demo = gr.Interface(
    fn=answer,
    inputs=[
        gr.Textbox(label="Verilog specification", lines=4,
                    placeholder="e.g. 8-to-1 multiplexer with 3-bit select"),
        gr.Slider(0.0, 1.2, value=0.2, step=0.05, label="Temperature (0 = deterministic, higher = more varied)"),
        gr.Number(value=1337, label="Seed", precision=0),
    ],
    outputs=gr.Code(label="Generated Verilog", language="c"),  # no verilog highlighter built in; c is close enough
    title="Verilog SLM",
    description="Type a spec, get a generated Verilog module back.",
)

# demo.launch(share=True, auth=("user", "pick-a-password"))  # uncomment for a password
demo.launch(share=True)

## Stopping

Interrupt this cell (or end the session) to take the page down. The
`share=True` link stops working immediately either way -- it's tunnelled
through this running process, not hosted independently.